In [1]:
import pandas as pd
import sqlite3

hourly_df = pd.read_csv('../data/processed/hourly_energy.csv')
hourly_df.head()

,timestamp,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,date,year,month,hour,day_of_week,is_weekend
0,2006-12-16 17:00:00,4.222889,0.229000,234.643889,18.100000,0.0,19.0,607.0,2006-12-16,2006,12,17,Saturday,True
1,2006-12-16 18:00:00,3.632200,0.080033,234.580167,15.600000,0.0,403.0,1012.0,2006-12-16,2006,12,18,Saturday,True
2,2006-12-16 19:00:00,3.400233,0.085233,233.232500,14.503333,0.0,86.0,1001.0,2006-12-16,2006,12,19,Saturday,True
3,2006-12-16 20:00:00,3.268567,0.075100,234.071500,13.916667,0.0,0.0,1007.0,2006-12-16,2006,12,20,Saturday,True
4,2006-12-16 21:00:00,3.056467,0.076667,237.158667,13.046667,0.0,25.0,1033.0,2006-12-16,2006,12,21,Saturday,True


In [2]:
conn = sqlite3.connect('../data/processed/energy_data.db')

hourly_df.to_sql('energy_consumption', conn, if_exists='replace', index=False)

print("Data loaded into SQLite successfully!")

Data loaded into SQLite successfully!


In [3]:
query = "SELECT * FROM energy_consumption LIMIT 5;"
pd.read_sql(query, conn)

,timestamp,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,date,year,month,hour,day_of_week,is_weekend
0,2006-12-16 17:00:00,4.222889,0.229000,234.643889,18.100000,0.0,19.0,607.0,2006-12-16,2006,12,17,Saturday,1
1,2006-12-16 18:00:00,3.632200,0.080033,234.580167,15.600000,0.0,403.0,1012.0,2006-12-16,2006,12,18,Saturday,1
2,2006-12-16 19:00:00,3.400233,0.085233,233.232500,14.503333,0.0,86.0,1001.0,2006-12-16,2006,12,19,Saturday,1
3,2006-12-16 20:00:00,3.268567,0.075100,234.071500,13.916667,0.0,0.0,1007.0,2006-12-16,2006,12,20,Saturday,1
4,2006-12-16 21:00:00,3.056467,0.076667,237.158667,13.046667,0.0,25.0,1033.0,2006-12-16,2006,12,21,Saturday,1


In [4]:
query = """
SELECT 
    ROUND(SUM(Global_active_power), 2) AS total_consumption_kwh,
    ROUND(AVG(Global_active_power), 2) AS avg_hourly_consumption_kwh,
    ROUND(MAX(Global_active_power), 2) AS max_hourly_consumption_kwh,
    ROUND(MIN(Global_active_power), 2) AS min_hourly_consumption_kwh
FROM energy_consumption;
"""
pd.read_sql(query, conn)

,total_consumption_kwh,avg_hourly_consumption_kwh,max_hourly_consumption_kwh,min_hourly_consumption_kwh
0,37302.15,1.09,6.56,0.12


In [5]:
query = """
SELECT 
    hour,
    ROUND(AVG(Global_active_power), 2) AS avg_consumption_kwh
FROM energy_consumption
GROUP BY hour
ORDER BY avg_consumption_kwh DESC
LIMIT 5;
"""
pd.read_sql(query, conn)

,hour,avg_consumption_kwh
0,20,1.90
1,21,1.88
2,19,1.73
3,7,1.50
4,8,1.46


In [6]:
query = """
SELECT 
    CASE WHEN is_weekend = 1 THEN 'Weekend' ELSE 'Weekday' END AS day_type,
    ROUND(AVG(Global_active_power), 2) AS avg_consumption_kwh
FROM energy_consumption
GROUP BY day_type;
"""
pd.read_sql(query, conn)

,day_type,avg_consumption_kwh
0,Weekday,1.04
1,Weekend,1.23


In [7]:
query = """
SELECT 
    month,
    ROUND(AVG(Global_active_power), 2) AS avg_consumption_kwh
FROM energy_consumption
GROUP BY month
ORDER BY avg_consumption_kwh DESC;
"""
pd.read_sql(query, conn)

,month,avg_consumption_kwh
0,12,1.49
1,1,1.46
2,2,1.30
3,11,1.29
4,3,1.23
5,10,1.14
6,4,1.05
7,5,1.03
8,9,0.98
9,6,0.91


In [8]:
query = """
SELECT 
    ROUND(SUM(Sub_metering_1), 2) AS kitchen_total,
    ROUND(SUM(Sub_metering_2), 2) AS laundry_total,
    ROUND(SUM(Sub_metering_3), 2) AS water_heater_ac_total
FROM energy_consumption;
"""
pd.read_sql(query, conn)

,kitchen_total,laundry_total,water_heater_ac_total
0,2299135.0,2661031.0,13235167.0
